## Notebook Summary

This notebook is a tutorial-style overview of the `tsim` package.

- It walks through supported gates, including Clifford gates (`H`, `S`, `CNOT`) and non-Clifford gates (`T`, `R_X`, `R_Y`, `R_Z`, `U3`).
- It covers measurement and reset features, measurement-record control, `MPP`, `MXX`/`MYY`/`MZZ`, `MPAD`, and `SPP`.
- It demonstrates noise channels, annotations, multiple samplers, visualization modes, and detector error model generation.
- The final example constructs a small noisy circuit and calls `c.detector_error_model()`, producing a `stim.DetectorErrorModel`.


# Overview


Tsim is a quantum circuit sampler that can efficiently sample from Clifford+T circuits with Pauli noise. It is based on ZX-calculus stabilizer rank decomposition and parametrized ZX diagrams, following work of [arXiv:2403.06777](https://arxiv.org/abs/2403.06777).

## Supported Gates

Tsim supports a universal gate set, together with measurement and reset instructions, and Pauli noise channels.

### Clifford Instructions

Tsim supports all instructions [supported by Stim](https://github.com/quantumlib/Stim/wiki/Stim-v1.13-Gate-Reference). Below, we show the standard generating set of Clifford gates:

In [1]:
from tsim import Circuit

c = Circuit("H 0")  # it maps $X$-errors to $Z$-errors and vice versa.
print(c.to_matrix())
c.diagram("timeline-svg", height=120)

[[ 0.70710678+0.j  0.70710678+0.j]
 [ 0.70710678+0.j -0.70710678+0.j]]


$$
H =
\frac{1}{\sqrt{2}}
\left(
\begin{array}{cc}
1 & 1 \\
1 & -1
\end{array}
\right)
$$

In [2]:
c = Circuit("S 0")  #  It maps $X$-errors to $Y$-errors while leaving $Z$-errors unchanged
print(c.to_matrix())
c.diagram("timeline-svg", height=120)

[[ 1.+0.j  0.-0.j]
 [-0.+0.j  0.+1.j]]


$$
S =
\left(
\begin{array}{cc}
1 & 0 \\
0 & i
\end{array}
\right)
$$

In [3]:
c = Circuit("CNOT 0 1")
print(c.to_matrix())
c.diagram("timeline-svg", height=160)

[[ 1.+0.j  0.+0.j -0.+0.j  0.+0.j]
 [ 0.+0.j  1.+0.j  0.+0.j -0.+0.j]
 [ 0.+0.j -0.+0.j  0.+0.j  1.+0.j]
 [-0.+0.j  0.+0.j  1.+0.j  0.+0.j]]


$$
\mathrm{CNOT} =
\left(
\begin{array}{cccc}
1 & 0 & 0 & 0 \\
0 & 1 & 0 & 0 \\
0 & 0 & 0 & 1 \\
0 & 0 & 1 & 0
\end{array}
\right)
$$

### Non-Clifford Instructions

In addition to Clifford gates, Tsim supports the following non-Clifford gates. Note that all rotation parameters are defined in units of $\pi$.

Computation time and memory requirement scales exponentially with the number of non-Clifford gates.

In [4]:
c = Circuit("T 0")  # Z-error unchanged, X,Y errors come out as liner combination of both
print(c.to_matrix())
c.diagram("timeline-svg", height=120)

[[ 1.        +0.j          0.        +0.j        ]
 [-0.        +0.j          0.70710678+0.70710678j]]


$$
T =
\left(
\begin{array}{cc}
1 & 0 \\
0 & e^{i\pi/4}
\end{array}
\right)
$$

In [5]:
c = Circuit("R_X(0.1) 0")  # rotation around X-axis by 0.1π
print(c.to_matrix())
c.diagram("timeline-svg", height=120)

[[ 0.98768834-0.j          0.        -0.15643447j]
 [-0.        -0.15643447j  0.98768834-0.j        ]]


$$
R_X(\alpha) =
\left(
\begin{array}{cc}
\cos(\alpha\pi/2) & -i \sin(\alpha\pi/2) \\
-i \sin(\alpha\pi/2) & \cos(\alpha\pi/2)
\end{array}
\right)
$$

In [6]:
c = Circuit("R_Y(0.1) 0")  # rotation around Y-axis by 0.1π
print(c.to_matrix())
c.diagram("timeline-svg", height=120)

[[ 0.98768834-0.j -0.15643447+0.j]
 [ 0.15643447-0.j  0.98768834-0.j]]


$$
R_Y(\alpha) =
\left(
\begin{array}{cc}
\cos(\alpha\pi/2) & -\sin(\alpha\pi/2) \\
\sin(\alpha\pi/2) & \cos(\alpha\pi/2)
\end{array}
\right)
$$

In [7]:
c = Circuit("R_Z(0.1) 0")  # rotation around Z-axis by 0.1π
print(c.to_matrix())
c.diagram("timeline-svg", height=120)

[[ 0.98768834-0.15643447j  0.        +0.j        ]
 [-0.        +0.j          0.98768834+0.15643447j]]


$$
R_Z(\alpha) =
\left(
\begin{array}{cc}
e^{-i\alpha\pi/2} & 0 \\
0 & e^{i\alpha\pi/2}
\end{array}
\right)
$$

In [8]:
c = Circuit("U3(0.1, 0.2, 0.3) 0")
print(c.to_matrix())
c.diagram("timeline-svg", height=120)

[[ 0.98768834-0.j         -0.09194987-0.12655814j]
 [ 0.12655814+0.09194987j  0.        +0.98768834j]]


$$
U_3(\theta, \phi, \lambda) =
\left(
\begin{array}{cc}
\cos(\theta\pi/2) & -e^{i\lambda\pi}\sin(\theta\pi/2) \\
e^{i\phi\pi}\sin(\theta\pi/2) & e^{i(\phi+\lambda)\pi}\cos(\theta\pi/2)
\end{array}
\right)
$$

### Measurement and Reset instructions

Tsim supports all collapsing gates [supported by Stim](https://github.com/quantumlib/Stim/wiki/Stim-v1.9-Gate-Reference#collapsing-gates).

In [9]:
c = Circuit("M 0")
c.diagram("timeline-svg", height=120)

Measurements (`M`, `MX`, `MY`, `MZ`) project the state into the measurement basis and write the resulting bit into the measurement record.

The measurement record can be used to conditionally apply Pauli gates:

In [10]:
c = Circuit("""
M 0
CY rec[-1] 1
""")
c.diagram("timeline-svg", height=170)

The `!` operator can be used to invert the classical measurement bit that is written into the measurement record:

In [11]:
c = Circuit("M !0")
c.diagram("timeline-svg", height=120)

In Stim, the MPP instruction stands for Measure Pauli Product.  It measures parity of joint state of qubits listed as argument. The result of MPP is a partially collapsed state, restricted to a smaller subspace, where this DOF was removed.

MPP safely extracts the error signature (the syndrome) without ever "looking" at the individual data qubits.

In actual quantum hardware, you cannot directly measure a Pauli string. If you want to know the combined parity of 4 data qubits, you have to allocate a separate ancilla qubit, perform 4 entangling CNOT gates, and then measure the ancilla.

The MPP instruction is Stim's "magic wand." It allows you to directly measure the joint parity of multiple qubits in a single step, bypassing the need for ancilla qubits, CNOT gates, or routing.

It collapses the quantum state into an eigenstate of that Pauli string and returns a single classical bit: 0 (if the parity is even/+1) or 1 (if the parity is odd/-1).

The `MPP` instruction measures Pauli strings. The MPP can also be used in conjunction with the `!` operator to flip the classical measurement bit before writing it into the measurement record.

In [12]:
c = Circuit("MPP !Z0*X2*Y3")
c.diagram("timeline-svg", height=260)

The `MXX`, `MYY`, and `MZZ` instructions measure two-qubit Pauli parity. For example, `MZZ` measures whether pairs of qubits are in the same-parity subspace ($\{|00\rangle, |11\rangle\}$) or opposite-parity subspace ($\{|01\rangle, |10\rangle\}$).

MZZ and MPP overlap in functionality, but they differ in flexibility, syntax, and scope.The short answer is: MPP Z0*Z1 and MZZ 0 1 do the exact same thing. They both perform a non-destructive quantum parity measurement of the $Z \otimes Z$ observable on two qubits. MZZ is essentially a specialized, shorthand version of MPP.

In [13]:
c = Circuit("MZZ 0 1")
c.diagram("timeline-svg", height=170)

The `MPAD` instruction pads the measurement record with fixed bit values (0 or 1), without performing any actual measurement.

MPAD stands for Measurement Pad. Usage:
1. Record Alignment for Decoders
When you simulate a circuit in Stim, the output is a giant 1D array of classical bits. Decoders (like PyMatching or your Machine Learning decoders) expect this array to be a very specific, rigid size so they know exactly which bit corresponds to which physical syndrome check.
2. Classical Flagging / Metadata
Sometimes you want to leave "breadcrumbs" in your measurement record for your post-processing scripts.

For example, if you are running 100 rounds of memory benchmarking followed by a logical state injection, you might inject MPAD 1 1 1 1 right between the two circuit blocks. When your Python script reads the output array, it can easily search for that 1111 sequence to know exactly where the memory rounds ended and the injection began, without having to calculate the exact index offsets manually.

In [14]:
c = Circuit("MPAD 0 1 1 0")
c.diagram("timeline-svg", height=170)

The `SPP` instruction phases the $-1$ eigenspace of a Pauli product observable by $i$. For example, `SPP Z0` is equivalent to the `S` gate, and `SPP X0*X1` is equivalent to `SQRT_XX`. The `SPP_DAG` instruction phases by $-i$ instead. The `!` operator inverts the product, swapping `SPP` and `SPP_DAG` behavior.

SPP stands for S-gate on a Pauli Product (and it comes with its inverse, SPP_DAG).
If MPP is a generalized abstract measurement, SPP is a generalized abstract Clifford rotation.

A standard S gate (the Phase gate) applies a 90-degree ($\pi/2$) rotation around the Z-axis of a single qubit.The SPP instruction generalizes this concept to multiple qubits. Instead of rotating around a single qubit's axis, it applies a 90-degree rotation around the axis defined by an entire Pauli string.

For example: **SPP X0\*Y1\*Z2**
 Applies a 90-degree rotation around the joint X0*Y1*Z2 axis


Mathematically, this applies the unitary $e^{-i \frac{\pi}{4} (X_0 \otimes Y_1 \otimes Z_2)}$. If a quantum error (or another Pauli operator) propagates through this gate and anti-commutes with that string, it gets rotated by 90 degrees in the Pauli frame.

In [15]:
c = Circuit("SPP X0*Y1*Z2")
c.diagram("timeline-svg", height=220)

### Noise Channels

Tsim supports all noise channels [supported by Stim](https://github.com/quantumlib/Stim/wiki/Stim-v1.9-Gate-Reference#noise-channels).

The `X_ERROR(p)` instruction is a `X` instruction that is applied with probability `p`.

In [16]:
c = Circuit("X_ERROR(0.1) 0")
c.diagram("timeline-svg", height=120)

The `PAULI_CHANNEL_1(p_x, p_y, p_z)` instruction is a `X`, `Y`, and `Z` instruction that is applied with probabilities `p_x`, `p_y`, and `p_z` respectively.

In [17]:
c = Circuit("PAULI_CHANNEL_1(0.1, 0.2, 0.3) 0")
c.diagram("timeline-svg", height=120)

The `PAULI_CHANNEL_2` instruction takes fifteen floats specifying the disjoint probabilities of each possible Pauli pair
that can occur (except for the non-error double identity case).
The disjoint probability arguments are (in order):

p_ix,
p_iy,
p_iz,
p_xi,
p_xx,
p_xy,
p_xz,
p_yi,
p_yx,
p_yy,
p_yz,
p_zi,
p_zx,
p_zy,
p_zz

In [18]:
c = Circuit(
    "PAULI_CHANNEL_2(0.01, 0.01, 0.03, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01) 0 1"
)
c.diagram("timeline-svg", height=170)

The `DEPOLARIZE1(p)` instruction applies a randomly chosen Pauli with probability `p`.

the difference between X_ERROR and DEPOLARIZE1 comes down to the difference between biased noise and symmetric (white) noise.

Both instructions inject probabilistic errors into your circuit, but they behave completely differently in how they distribute those errors across the Pauli axes.

In [19]:
c = Circuit("DEPOLARIZE1(0.01) 0")
c.diagram("timeline-svg", height=120)

The `DEPOLARIZE2(p)` instruction applies a randomly chosen two-qubit Pauli with probability `p`.

In [20]:
c = Circuit("DEPOLARIZE2(0.01) 0 1")
c.diagram("timeline-svg", height=170)

The `CORRELATED_ERROR(p)` instruction applies a specified Pauli product with probability `p`. If no error occurred, then a following `ELSE_CORRELATED_ERROR(p2)` instruction may apply a Pauli product with probabiliy `p2`. If no error occurs again, further `ELSE_CORRELATED_ERROR(pi)` instructions in the chain may apply a Pauli products.

In Stim, the CORRELATED_ERROR instruction is how you model crosstalk and multi-qubit faults.While X_ERROR and DEPOLARIZE1 apply noise to qubits independently, CORRELATED_ERROR(p) allows you to inject a specific Pauli string of errors across multiple qubits simultaneously, with a single shared probability $p$.

In [21]:
c = Circuit("""
CORRELATED_ERROR(0.1) X0  # Apply X with probability 0.1
ELSE_CORRELATED_ERROR(0.2) Z0 Y1  # If no error occurred, apply Z0*Y1 with probability 0.2
ELSE_CORRELATED_ERROR(0.3) X1 # If still no error, apply X1 with probability 0.3
""")
c.diagram("timeline-svg", height=170)

### Annotations

Tsim supports detector and observable annotations.

The `DETECTOR` instruction is only used in detector sampling mode and ignored otherwise. It instructs the detector sampler to record the XOR of classical outcomes of specified measurement bits.

In Stim, the DETECTOR instruction is arguably the most important feature for Quantum Error Correction (QEC) researchers.While instructions like M or MPP generate the raw, classical measurement data, DETECTOR is a classical annotation. It does not affect the quantum state at all. Instead, it tells the simulator how to interpret those measurements to find errors.

Here is the breakdown of what it is, how it works, and why your Machine Learning decoders absolutely depend on it:

1. The Core Concept: The "Zero-Expectation" AlarmIn a perfect, noiseless quantum computer, the parity checks (stabilizers) of a QEC code are completely deterministic.
2. If you prepare a $|0_L\rangle$ state and measure its stabilizers, you know exactly what the answers should be.
3. If you measure a stabilizer in Round 1, and then measure that exact same stabilizer in Round 2, the outcome should be identical.
  
   A DETECTOR is an instruction you place in your Stim circuit to declare: "Under noiseless conditions, the parity of these specific measurements must be EVEN (0)."If Stim runs the circuit with noise, and that parity flips to ODD (1), the detector "fires." This is called a detection event or a defect.

In [22]:
c = Circuit("""
    M 0 0
    DETECTOR rec[-1] rec[-2]
""")
c.diagram("timeline-svg", height=150)

The `OBSERVABLE_INCLUDE` instruction is only used in observable sampling mode and ignored otherwise. It instructs the detector sampler to record the XOR of the specified measurement bits.

In Stim, while DETECTOR tracks the local errors (the clues), OBSERVABLE_INCLUDE tracks the global logical state (the final answer).

It is the mechanism you use to define your logical qubits and, ultimately, it is the grading rubric that tells your decoder whether its error corrections succeeded or failed.

In a QEC code, a single logical qubit is made up of many physical qubits. To "read" the logical state (for example, to measure the logical $Z_L$ operator), you have to measure a specific string of physical qubits and calculate their joint parity.For the Steane [[7, 1, 3]] code, the logical $Z_L$ operator is the joint $Z$-parity of all 7 physical data qubits.If you physically measure all 7 data qubits at the end of your circuit, you use OBSERVABLE_INCLUDE to tell Stim: "Take the parity of these 7 specific measurements. This parity represents my logical state."

In [23]:
c = Circuit("""
    M 0 0
    OBSERVABLE_INCLUDE(0) rec[-1] rec[-2]
""")
c.diagram("timeline-svg", height=150)

## Sampling

Tsim supports multiple samplers. The first is a measurement sampler. This will simply sample bits for each measurement instruction in the circuit. Detector and observable annotations will simply be ignored by this sampler.

In [24]:
c = Circuit("""
    RX 0    # Reset qubit 0 in the X-basis
    R 1     # reset qubit 1 back to the standard computational 0 state
    CNOT 0 1
    M 0 1
""")
sampler = c.compile_sampler()
c.diagram("timeline-svg", height=170)

In [25]:
c.diagram("pyzx", scale_horizontally=2);

In [26]:
sampler.sample(shots=5)

array([[ True,  True],
       [ True,  True],
       [ True,  True],
       [ True,  True],
       [False, False]])

The second sampling mode is detector sampling. This will sample detector events and observable values. Detector and observable bits can always be obtained by linear transformations of the measurement bits as return by the measurement sampler.
In practice, however, it can be much more efficient to sample detector events directly.

In [27]:
c = Circuit("""
    RX 0
    R 1
    CNOT 0 1
    M 0 1
    DETECTOR rec[-1] rec[-2]  # The Error Alarm, the answer is always 0 beacur only 00 or 11 are measured for bell state
    OBSERVABLE_INCLUDE(0) rec[-2] # The Data Payload, value of measurement indexted by [-j] stored in memory (aka logical qubit) '0'
""")
sampler = c.compile_detector_sampler()
c.diagram("timeline-svg", height=170)

In [28]:
detectors, observables = sampler.sample(5, separate_observables=True)
print(detectors)
print(observables)

[[False]
 [False]
 [False]
 [False]
 [False]]
[[ True]
 [ True]
 [False]
 [ True]
 [False]]


Finally, Tsim allows to compute probability values for target states via the `CompiledStateProbs` sampler.

In [29]:
import numpy as np
from tsim.sampler import CompiledStateProbs

sampler = CompiledStateProbs(c)

In [30]:
sampler.probability_of(np.array([0, 0]), batch_size=1)

array([0.5], dtype=float32)

In [31]:
# same infor but as a nicer table
import itertools

# Generate all 2-qubit states: [0,0], [0,1], [1,0], [1,1]
bitstrings = np.array(list(itertools.product([0, 1], repeat=2)))

print("q0 | q1 | Probability")
print("---|----|------------")

for state in bitstrings:
    # Extract the probability float from the Tsim array
    prob = sampler.probability_of(state, batch_size=1)[0]
    
    # Print as a formatted row
    print(f" {state[0]} |  {state[1]} | {prob:.2f}")

q0 | q1 | Probability
---|----|------------
 0 |  0 | 0.50
 0 |  1 | 0.00
 1 |  0 | 0.00
 1 |  1 | 0.50


## Visualization

Tsim supports multiple ways of visualizing quantum circuits.

The `timeline-svg` diagram shows the circuit as a time-ordered sequence of gates.

In [32]:
c = Circuit("""
    QUBIT_COORDS(0, 0) 0  # specifies qubit coordinates for the "timeslice-svg" diagram
    QUBIT_COORDS(1, 1) 1
    H 0
    TICK
    T 0
    H 0
    TICK
    CNOT 0 1
    TICK
    DEPOLARIZE2(0.1) 0 1
    TICK
    R_Z(0.2) 1
    TICK
    M 0 1
    DETECTOR rec[-1] rec[-2]
    OBSERVABLE_INCLUDE(0) rec[-1]
""")
c.diagram("timeline-svg", height=170)

When `TICK` instructions are present, each tick can be shown as a 2D time slice with the `timeslice-svg` diagram. Here, `QUBIT_COORDS` annotations can be used to specify the 2D coordinates of the qubits.

In [33]:
c.diagram("timeslice-svg", width=800, rows=1)

With the `pyzx` argument, the circuit can be visualized using the [pyzx](https://github.com/zxcalc/pyzx) as a ZX-diagram.

In [34]:
c.diagram("pyzx");

The `pyzx-meas` and `pyzx-dets` diagrams show ZX diagrams where outputs represent probabilities of measurement outcomes for measurement and detector/observables, respectively.

In [35]:
c.diagram("pyzx-meas", scale_horizontally=2);

In [36]:
c.diagram("pyzx-dets", scale_horizontally=1.5);

## Detector Error Models

Tsim allows to compute detector error models from a circuit. The method `Circuit.detector_error_model()` computes a `stim.DetectorErrorModel` from the circuit. As opposed to Stim, detectors and observables need not be deterministic.

In [37]:
c = Circuit("""
    RX 0
    R 1
    CNOT 0 1
    DEPOLARIZE1(0.1) 0 1
    M 0 1
    DETECTOR rec[-1] rec[-2]
    DETECTOR rec[-1]
""")
c.detector_error_model()

stim.DetectorErrorModel('''
    error(0.0666667) D0
    error(0.0666667) D0 D1
    error(0.5) D1
''')